## ⚠️ Important — Do Not Rerun This Notebook
This notebook pulls additional features from the OpenAlex API for papers already 
in the cleaned dataset. If rerun, results may differ as OpenAlex updates continuously.

**Date of enriched data pull:** 25.03.2026  
**Input:** `data/OpenAlex/openalex_ai_papers_cleaned.csv`  
**Output:** `data/OpenAlex/openalex_ai_papers_enriched.csv`  

# Enriched Data Pull

In [29]:
import pandas as pd
import numpy as np
from pyalex import Works
import pyalex
from dotenv import load_dotenv
import os

load_dotenv()
pyalex.config.email = os.getenv("OPENALEX_EMAIL")
pyalex.config.api_key = os.getenv("OPENALEX_API_KEY")
pyalex.config.max_retries = 3
pyalex.config.retry_backoff_factor = 0.5
pyalex.config.retry_http_codes = [429, 500, 503]

## Load cleanded data and extract IDs

In [30]:
# Load cleaned dataset — we will filter pull results to only these IDs
df_cleaned = pd.read_csv('../data/OpenAlex/openalex_ai_papers_cleaned.csv')
cleaned_ids = set(df_cleaned['id'].tolist())

print(f'Papers in cleaned dataset: {len(cleaned_ids):,}')
print(f'Years: {df_cleaned["publication_year"].min()} — {df_cleaned["publication_year"].max()}')
print(f'Topics: {df_cleaned["topic_name"].nunique()}')

Papers in cleaned dataset: 293,045
Years: 2015 — 2024
Topics: 10


## Topics defiition

In [31]:
# Same topics as original pull
topics = {
    "T10181": "Natural Language Processing Techniques",
    "T10320": "Neural Networks and Applications",
    "T10028": "Topic Modeling",
    "T10201": "Speech Recognition and Synthesis",
    "T10664": "Sentiment Analysis and Opinion Mining",
    "T11512": "Anomaly Detection Techniques and Applications",
    "T11975": "Evolutionary Algorithms and Applications",
    "T10100": "Metaheuristic Optimization Algorithms Research",
    "T10764": "Privacy-Preserving Technologies in Data",
    "T10682": "Quantum Computing Algorithms and Architecture",
}

## Helper functions

In [33]:
def parse_authorships(authorships):
    """
    Extract authorship-related features from the nested authorships field.
    
    Note on author counting: some older papers have author.id = None even though
    the author exists. In these cases we count authorship entries directly instead
    of relying on IDs. Where IDs are available we use them for deduplication.
    
    Returns tuple of:
    - unique_authors_count: number of distinct authors
    - unique_institutions_count: number of distinct institution IDs
    - inst_edu: number of education institutions
    - inst_nonprofit: number of nonprofit institutions
    - inst_gov: number of government institutions
    - inst_company: number of company institutions
    """
    if not isinstance(authorships, list):
        return 0, 0, 0, 0, 0, 0
    
    author_ids = set()
    author_count = 0
    seen_inst_ids = set()
    inst_edu = inst_nonprofit = inst_gov = inst_company = 0
    
    for a in authorships:
        # Count every authorship entry
        author_count += 1
        # Also collect IDs where available for deduplication
        if isinstance(a.get('author'), dict) and a['author'].get('id'):
            author_ids.add(a['author']['id'])
        
        # Unique institutions and type breakdown
        # Using seen_inst_ids to avoid double counting institutions
        # that appear across multiple authors
        for inst in a.get('institutions', []) or []:
            inst_id = inst.get('id')
            inst_type = inst.get('type', '')
            
            if inst_id and inst_id not in seen_inst_ids:
                seen_inst_ids.add(inst_id)
                if inst_type == 'education': inst_edu += 1
                elif inst_type == 'nonprofit': inst_nonprofit += 1
                elif inst_type == 'government': inst_gov += 1
                elif inst_type == 'company': inst_company += 1
    
    # Use ID-based count where available, fall back to entry count
    unique_authors = len(author_ids) if len(author_ids) > 0 else author_count
    
    return (
        unique_authors,
        len(seen_inst_ids),
        inst_edu, inst_nonprofit, inst_gov, inst_company
    )


def parse_sdgs(sdgs):
    """
    Extract SDG-related features from the nested sustainable_development_goals field.
    
    Returns tuple of:
    - sdg_count: number of SDGs tagged
    - sdg_max_score: highest confidence score among tagged SDGs
    - sdg_avg_score: average confidence score across all tagged SDGs
    - sdg_display_names: list of SDG names for EDA
    - sdg_numbers: list of SDG numbers (1-17) for one-hot encoding later
    """
    if not isinstance(sdgs, list) or len(sdgs) == 0:
        return 0, 0, 0, [], []
    return (
        len(sdgs),
        max(s['score'] for s in sdgs),
        round(sum(s['score'] for s in sdgs) / len(sdgs), 4),
        # Display names kept for EDA — not used as model features directly
        [s['display_name'] for s in sdgs if s.get('display_name')],
        # Extract SDG number from URL e.g. https://metadata.un.org/sdg/4 → 4
        [int(s['id'].split('/')[-1]) for s in sdgs
         if s.get('id') and s['id'].split('/')[-1].isdigit()]
    )

## Pull function

In [34]:
def pull_topic_enriched(topic_id, topic_name, cleaned_ids, year_start=2015, year_end=2024):
    """
    Pull enriched features for a single topic across all years.
    Uses same structure as original pull — filter by topic/year in batches of 200,
    then keep only papers that exist in the cleaned dataset.
    """
    all_records = []
    
    for year in range(year_start, year_end + 1):
        
        # Pull only the new fields needed for enrichment
        # id is included as the join key for merging with cleaned dataset later
        query = (
            Works()
            .filter(
                primary_topic={"id": topic_id},
                publication_year=str(year),
                type="article",
                is_retracted=False
            )
            .select([
                "id",
                "authorships",                   # unique authors, institution count and type
                "funders",                        # funder count
                "awards",                         # individual grant count
                "sustainable_development_goals",  # SDG alignment
                "referenced_works",               # raw list for scale estimation
            ])
        )
        
        records = []
        for page in query.paginate(per_page=200, n_max=None):
            records.extend(page)
        
        # Filter to only papers that survived the cleaning step
        records = [r for r in records if r['id'] in cleaned_ids]
        all_records.extend(records)
        print(f"  {year}: {len(records):,} papers")
    
    df = pd.DataFrame(all_records)
    
    # === FLATTEN AUTHORSHIPS ===
    # parse_authorships returns a tuple — unpack into separate columns
    # Note: countries removed — recalculated count showed no improvement
    # over existing countries_distinct_count in cleaned dataset
    authorship_parsed = df['authorships'].apply(parse_authorships)
    df['unique_authors_count']         = authorship_parsed.apply(lambda x: x[0])
    df['unique_institutions_count']    = authorship_parsed.apply(lambda x: x[1])
    df['institution_edu_count']        = authorship_parsed.apply(lambda x: x[2])
    df['institution_nonprofit_count']  = authorship_parsed.apply(lambda x: x[3])
    df['institution_gov_count']        = authorship_parsed.apply(lambda x: x[4])
    df['institution_company_count']    = authorship_parsed.apply(lambda x: x[5])
    
    # === FLATTEN FUNDERS ===
    # funder_count: number of distinct funding organisations
    df['funder_count'] = df['funders'].apply(
        lambda x: len(x) if isinstance(x, list) else 0)
    
    # === FLATTEN AWARDS ===
    # award_count: number of individual grants — different from funder_count
    # (same funder can give multiple grants)
    df['award_count'] = df['awards'].apply(
        lambda x: len(x) if isinstance(x, list) else 0)
    
    # === FLATTEN SDGs ===
    # parse_sdgs returns a tuple — unpack into separate columns
    sdg_parsed = df['sustainable_development_goals'].apply(parse_sdgs)
    df['sdg_count']         = sdg_parsed.apply(lambda x: x[0])
    df['sdg_max_score']     = sdg_parsed.apply(lambda x: x[1])
    df['sdg_avg_score']     = sdg_parsed.apply(lambda x: x[2])
    df['sdg_display_names'] = sdg_parsed.apply(lambda x: x[3])
    df['sdg_numbers']       = sdg_parsed.apply(lambda x: x[4])
    
    # === REFERENCED WORKS ===
    # Keep raw list for scale estimation in Cell 7
    # Will be dropped before saving to CSV
    df['referenced_works_list'] = df['referenced_works'].apply(
        lambda x: x if isinstance(x, list) else [])
    
    # Drop raw nested columns — all information extracted into flat columns above
    df = df.drop(columns=[
        'authorships', 'funders', 'awards',
        'sustainable_development_goals', 'referenced_works'
    ])
    
    print(f"✅ {topic_name}: {len(df):,} papers total\n")
    return df

## Run Pull

In [35]:
dfs = []

for topic_id, topic_name in topics.items():
    df = pull_topic_enriched(topic_id, topic_name, cleaned_ids)
    dfs.append(df)

# Concatenate all topics into one DataFrame
df_enriched = pd.concat(dfs, ignore_index=True)

print(f"\n📊 Total records pulled: {len(df_enriched):,}")
print(f"📊 Papers in cleaned dataset: {len(cleaned_ids):,}")
# Match rate should be close to 100% — low rate indicates ID mismatch issue
print(f"📊 Match rate: {len(df_enriched)/len(cleaned_ids)*100:.1f}%")

  2015: 5,412 papers
  2016: 5,715 papers
  2017: 5,085 papers
  2018: 5,444 papers
  2019: 5,482 papers
  2020: 5,704 papers
  2021: 4,738 papers
  2022: 4,081 papers
  2023: 4,829 papers
  2024: 7,286 papers
✅ Natural Language Processing Techniques: 53,776 papers total

  2015: 2,925 papers
  2016: 3,152 papers
  2017: 3,173 papers
  2018: 3,694 papers
  2019: 3,992 papers
  2020: 3,507 papers
  2021: 3,046 papers
  2022: 2,309 papers
  2023: 2,692 papers
  2024: 6,007 papers
✅ Neural Networks and Applications: 34,497 papers total

  2015: 1,767 papers
  2016: 2,131 papers
  2017: 2,442 papers
  2018: 3,449 papers
  2019: 4,436 papers
  2020: 5,395 papers
  2021: 6,442 papers
  2022: 6,417 papers
  2023: 8,654 papers
  2024: 7,611 papers
✅ Topic Modeling: 48,744 papers total

  2015: 1,774 papers
  2016: 1,880 papers
  2017: 1,695 papers
  2018: 1,879 papers
  2019: 2,033 papers
  2020: 2,020 papers
  2021: 2,083 papers
  2022: 2,175 papers
  2023: 2,592 papers
  2024: 3,042 papers
✅

## Estimate referenced works scale and funders

In [36]:
# Collect all unique referenced work IDs across the entire dataset
# This tells us whether looking up citation counts per referenced work is feasible
all_ref_ids = set()
for refs in df_enriched['referenced_works_list']:
    all_ref_ids.update(refs)

print(f'Total unique referenced work IDs: {len(all_ref_ids):,}')
print(f'Average references per paper: {df_enriched["referenced_works_list"].apply(len).mean():.1f}')
print(f'Max references per paper: {df_enriched["referenced_works_list"].apply(len).max():,}')
print(f'\nIf unique IDs < 500k — lookup is feasible')
print(f'If unique IDs > 500k — lookup will take too long, skip this feature')

Total unique referenced work IDs: 1,705,251
Average references per paper: 24.4
Max references per paper: 911

If unique IDs < 500k — lookup is feasible
If unique IDs > 500k — lookup will take too long, skip this feature


Unique referenced work ids is almost 2 million, too large to pull average citation score. We will drop this column

In [38]:
# Check null counts and percentage across all enriched columns
null_counts = df_enriched.isnull().sum()
null_pct = (df_enriched.isnull().sum() / len(df_enriched) * 100).round(2)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct
}).sort_values('null_pct', ascending=False)

print(null_summary)

                             null_count  null_pct
id                                    0       0.0
unique_authors_count                  0       0.0
unique_institutions_count             0       0.0
institution_edu_count                 0       0.0
institution_nonprofit_count           0       0.0
institution_gov_count                 0       0.0
institution_company_count             0       0.0
funder_count                          0       0.0
award_count                           0       0.0
sdg_count                             0       0.0
sdg_max_score                         0       0.0
sdg_avg_score                         0       0.0
sdg_display_names                     0       0.0
sdg_numbers                           0       0.0
referenced_works_list                 0       0.0


In [39]:
# Check zero counts across all numerical columns before saving
numerical_enriched = [
    'unique_authors_count', 'unique_institutions_count',
    'institution_edu_count', 'institution_nonprofit_count',
    'institution_gov_count', 'institution_company_count',
    'funder_count', 'award_count',
    'sdg_count', 'sdg_max_score', 'sdg_avg_score'
]

zero_summary = pd.DataFrame({
    'zero_count': (df_enriched[numerical_enriched] == 0).sum(),
    'zero_pct': ((df_enriched[numerical_enriched] == 0).sum() / len(df_enriched) * 100).round(2)
}).sort_values('zero_pct', ascending=False)

print(zero_summary)

                             zero_count  zero_pct
institution_nonprofit_count      288591     98.49
institution_gov_count            286664     97.84
institution_company_count        270043     92.16
award_count                      237333     81.00
funder_count                     216413     73.86
sdg_max_score                    143244     48.89
sdg_count                        143244     48.89
sdg_avg_score                    143244     48.89
institution_edu_count             89817     30.65
unique_institutions_count         73703     25.15
unique_authors_count               1694      0.58


In [40]:
# Check papers with 0 unique authors
print("Papers with 0 unique authors:")
print(df_enriched[df_enriched['unique_authors_count'] == 0][['id', 'unique_authors_count', 'unique_institutions_count']].head(10))

print(f"\nTotal papers with 0 authors: {(df_enriched['unique_authors_count'] == 0).sum():,}")
print(f"Total papers with 0 institutions: {(df_enriched['unique_institutions_count'] == 0).sum():,}")

Papers with 0 unique authors:
                                    id  unique_authors_count  \
195   https://openalex.org/W4302408502                     0   
1008  https://openalex.org/W2209419522                     0   
1112  https://openalex.org/W1815708336                     0   
2002  https://openalex.org/W4299588831                     0   
2499  https://openalex.org/W4249825799                     0   
3011  https://openalex.org/W4234770864                     0   
5182  https://openalex.org/W4230631453                     0   
5185  https://openalex.org/W4231919508                     0   
5188  https://openalex.org/W4232633708                     0   
5191  https://openalex.org/W4233853168                     0   

      unique_institutions_count  
195                           0  
1008                          0  
1112                          0  
2002                          0  
2499                          0  
3011                          0  
5182                       

In [44]:
# Check if institution zeros correlate with author ID nulls
zero_inst_has_authors = df_enriched[
    (df_enriched['unique_institutions_count'] == 0) & 
    (df_enriched['unique_authors_count'] > 0)
]
print(f"Papers with 0 institutions but authors > 0: {len(zero_inst_has_authors):,}")

Papers with 0 institutions but authors > 0: 72,009


## save enriched csv

In [46]:
# Drop columns not needed as model features:
# - referenced_works_list: raw ID list, scale too large for citation lookup (1.7M unique IDs)
# - institution_nonprofit_count: 98.5% zeros — too sparse to be useful
# - institution_gov_count: 97.8% zeros — too sparse to be useful
# - institution_company_count: 92.2% zeros — too sparse to be useful
df_save = df_enriched.drop(columns=[
    'referenced_works_list',
    'institution_nonprofit_count',
    'institution_gov_count',
    'institution_company_count'
])

df_save.to_csv('../data/OpenAlex/openalex_ai_papers_enriched.csv', index=False)
print(f'✅ Saved: {df_save.shape[0]:,} rows, {df_save.shape[1]} columns')
df_save.columns.tolist()

✅ Saved: 293,002 rows, 11 columns


['id',
 'unique_authors_count',
 'unique_institutions_count',
 'institution_edu_count',
 'funder_count',
 'award_count',
 'sdg_count',
 'sdg_max_score',
 'sdg_avg_score',
 'sdg_display_names',
 'sdg_numbers']

## Next Steps

### Funder h-index lookup — decided to skip
74% of papers have no funder data in OpenAlex — too sparse to add meaningful signal.
`funder_count` and `award_count` are kept as simpler funder features.

### Referenced works lookup — decided to skip  
1.7M unique referenced work IDs — lookup not feasible within reasonable time.
`referenced_works_count` from the cleaned dataset already captures quantity signal.

### Proceed to enriched feature engineering
Merge `openalex_ai_papers_enriched.csv` with
`openalex_ai_papers_cleaned.csv` on `id` and proceed
to `05_AI_Papers_feature_engineering_enriched.ipynb`